# 01 Data Processing

This notebook processes the country-year panel from the Excel workbooks available in `data/raw`. The workflow follows a systematic missingness policy:

- diagnose missingness before filling anything,
- keep the dependent variable and key independent variables on observed data only,
- interpolate eligible control variables within country over time,
- leave structural gaps unfilled,
- export a slim analysis-ready panel plus explicit missingness diagnostics.


## Table Of Contents

- [01 Setup And Variable Contract](#01-Setup-And-Variable-Contract)
- [02 Load Source Panel](#02-Load-Source-Panel)
- [03 Verify Existing Transformations](#03-Verify-Existing-Transformations)
- [04 Missingness Diagnostics](#04-Missingness-Diagnostics)
- [05 Role-Based Missingness Handling](#05-Role-Based-Missingness-Handling)
- [06 Panel Balance And Sample Checks](#06-Panel-Balance-And-Sample-Checks)
- [07 Export Processing Artifacts](#07-Export-Processing-Artifacts)


## 01 Setup And Variable Contract

Define paths, expected columns, variable roles, and notebook-wide constants.

### 01.1 Paths And Imports

Set the project paths and display defaults before any data decisions are made.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import (
    OUTPUTS_DIR,
    PRIMARY_WORKBOOK,
    PROCESSED_DIR,
    RAW_DIR,
    WINSOR_BOUNDS,
    ensure_output_dirs,
)

ensure_output_dirs()
RAW_EXCEL_FILES = sorted(RAW_DIR.glob('*.xlsx'))

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

### 01.2 Variable Contract

Define the dependent variable, monetary-policy variables, controls, and model-ready output columns. This is the contract used by later missingness and modeling checks.


In [2]:
BASE_COLUMNS = [
    'country_id',
    'country_code',
    'country',
    'year',
    'fdi_pct_gdp',
    'broad_money_pct_gdp',
    'trade_pct_gdp',
    'population_total',
    'inflation_gdp_deflator_pct',
    'official_exchange_rate_lcu_usd',
    'gdppc_current_usd',
    'deposit_interest_rate_pct',
    'tourism_arrivals',
    'hc_human_capital_index',
    'ln_gdppc',
    'xr_dep_pct',
]
OPTIONAL_RAW_COLUMNS = [
    'real_interest_rate_pct',
    'lending_interest_rate_pct',
]
EXPECTED_COLUMNS = BASE_COLUMNS + OPTIONAL_RAW_COLUMNS

DEP_VAR = 'fdi_pct_gdp'
KEY_INDEPENDENT_VARS = [
    'broad_money_pct_gdp',
    'deposit_interest_rate_pct',
    'real_interest_rate_pct',
    'lending_interest_rate_pct',
]
CONTROL_VARS = [
    'trade_pct_gdp',
    'inflation_gdp_deflator_pct',
    'ln_gdppc',
    'xr_dep_pct',
    'ln_population_total',
    'ln_tourism_arrivals',
    'hc_human_capital_index',
]
NON_INTERPOLATABLE_CONTROLS = {
    'hc_human_capital_index',
}
WINSORIZED_SENSITIVITY_COLUMNS = [
    'fdi_pct_gdp_winsorized',
    'xr_dep_pct_winsorized',
]
CLEAN_PANEL_COLUMNS = [
    'country_id',
    'country_code',
    'country',
    'year',
    'fdi_pct_gdp',
    'fdi_pct_gdp_winsorized',
    'broad_money_pct_gdp',
    'trade_pct_gdp',
    'inflation_gdp_deflator_pct',
    'deposit_interest_rate_pct',
    'real_interest_rate_pct',
    'lending_interest_rate_pct',
    'hc_human_capital_index',
    'ln_gdppc',
    'xr_dep_pct',
    'xr_dep_pct_winsorized',
    'ln_population_total',
    'ln_tourism_arrivals',
]
INTEREST_RATE_SERIES_MAP = {
    'Real interest rate (%)': 'real_interest_rate_pct',
    'Lending interest rate (%)': 'lending_interest_rate_pct',
    'Deposit interest rate (%)': 'deposit_interest_rate_pct',
}
MECHANISM_PREDICTORS = [
    'ln_gdppc',
    'inflation_gdp_deflator_pct',
    'trade_pct_gdp',
    'broad_money_pct_gdp',
]

## 02 Load Source Panel

Read the workbook panel, normalize columns, and inspect the raw analysis frame.

In [3]:
def load_base_panel() -> pd.DataFrame:
    if not PRIMARY_WORKBOOK.exists():
        raise FileNotFoundError(
            'Expected a base panel workbook at '
            f'{PRIMARY_WORKBOOK}. The processing workflow uses the merged workbook as the base panel.'
        )
    df = pd.read_excel(PRIMARY_WORKBOOK, sheet_name='Clean_Data')
    for column in OPTIONAL_RAW_COLUMNS:
        if column not in df.columns:
            df[column] = np.nan
    missing = [column for column in EXPECTED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f'Base workbook is missing required columns: {missing}')
    return df[EXPECTED_COLUMNS].copy()


def extract_wdi_interest_rates(raw_excel_files: list[Path]) -> pd.DataFrame:
    panels = []
    for path in raw_excel_files:
        try:
            xls = pd.ExcelFile(path)
        except Exception:
            continue
        if 'Data' not in xls.sheet_names:
            continue
        try:
            raw = pd.read_excel(path, sheet_name='Data')
        except Exception:
            continue
        if not {'Country Code', 'Series Name'}.issubset(raw.columns):
            continue
        mask = raw['Series Name'].isin(INTEREST_RATE_SERIES_MAP)
        if not mask.any():
            continue
        year_columns = [column for column in raw.columns if '[YR' in str(column)]
        if not year_columns:
            continue
        long_df = raw.loc[mask, ['Country Code', 'Series Name', *year_columns]].melt(
            id_vars=['Country Code', 'Series Name'],
            value_vars=year_columns,
            var_name='year_label',
            value_name='value',
        )
        long_df['year'] = long_df['year_label'].astype(str).str.extract(r'(\d{4})').astype(int)
        long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce')
        long_df['variable'] = long_df['Series Name'].map(INTEREST_RATE_SERIES_MAP)
        panel = long_df.pivot_table(
            index=['Country Code', 'year'],
            columns='variable',
            values='value',
            aggfunc='first',
        ).reset_index().rename(columns={'Country Code': 'country_code'})
        panel.columns.name = None
        panels.append(panel)
    if not panels:
        return pd.DataFrame(columns=['country_code', 'year', *INTEREST_RATE_SERIES_MAP.values()])
    combined = pd.concat(panels, ignore_index=True)
    combined = combined.sort_values(['country_code', 'year']).drop_duplicates(['country_code', 'year'], keep='last')
    return combined.reset_index(drop=True)


df = load_base_panel()
interest_rates = extract_wdi_interest_rates(RAW_EXCEL_FILES)
if not interest_rates.empty:
    df = df.drop(columns=[column for column in OPTIONAL_RAW_COLUMNS if column in df.columns]).merge(
        interest_rates,
        on=['country_code', 'year'],
        how='left',
    )
for column in OPTIONAL_RAW_COLUMNS:
    if column not in df.columns:
        df[column] = np.nan

for column in ['country_id', 'year']:
    df[column] = pd.to_numeric(df[column], errors='coerce').astype('Int64')
for column in [column for column in EXPECTED_COLUMNS if column not in {'country_id', 'country_code', 'country', 'year'}]:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df = df.sort_values(['country', 'year']).reset_index(drop=True)
pd.DataFrame({'raw_workbook': [path.name for path in RAW_EXCEL_FILES]})


,raw_workbook
0,P_Data_Extract_From_World_Development_Indicato...
1,asean_fdi_monetary_policy_clean_merged.xlsx


## 03 Verify Existing Transformations

Recompute core transformed variables and check whether workbook values match the model-ready definitions.

In [4]:
df['ln_gdppc_recomputed'] = np.where(df['gdppc_current_usd'] > 0, np.log(df['gdppc_current_usd']), np.nan)
df['xr_dep_pct_recomputed'] = (
    df.groupby('country')['official_exchange_rate_lcu_usd']
    .transform(lambda series: np.log(series).diff() * 100)
)
df['ln_gdppc_diff'] = df['ln_gdppc'] - df['ln_gdppc_recomputed']
df['xr_dep_pct_diff'] = df['xr_dep_pct'] - df['xr_dep_pct_recomputed']
df['ln_population_total'] = np.where(df['population_total'] > 0, np.log(df['population_total']), np.nan)
df['ln_tourism_arrivals'] = np.where(df['tourism_arrivals'] > 0, np.log(df['tourism_arrivals']), np.nan)

verification_table = pd.DataFrame(
    {
        'metric': ['ln_gdppc', 'xr_dep_pct'],
        'comparable_rows': [
            int((df['ln_gdppc'].notna() & df['ln_gdppc_recomputed'].notna()).sum()),
            int((df['xr_dep_pct'].notna() & df['xr_dep_pct_recomputed'].notna()).sum()),
        ],
        'max_abs_diff': [
            float(df['ln_gdppc_diff'].abs().dropna().max()),
            float(df['xr_dep_pct_diff'].abs().dropna().max()),
        ],
        'status': [
            'match' if df['ln_gdppc_diff'].abs().dropna().max() < 1e-9 else 'review',
            'match' if df['xr_dep_pct_diff'].abs().dropna().max() < 1e-9 else 'review',
        ],
    }
)
verification_table


,metric,comparable_rows,max_abs_diff,status
0,ln_gdppc,154,0.0000,match
1,xr_dep_pct,140,0.0000,match


### 03.1 Transformation Verification Result

Use this table to see whether workbook transformations match the notebook definitions.


In [5]:
display(verification_table)


,metric,comparable_rows,max_abs_diff,status
0,ln_gdppc,154,0.0000,match
1,xr_dep_pct,140,0.0000,match


## 04 Audit Missingness And Review Flags

Classify variable roles, coverage gaps, and values that need manual review before modeling.

In [6]:
def role_group(variable: str) -> str:
    if variable == DEP_VAR:
        return 'dependent'
    if variable in KEY_INDEPENDENT_VARS:
        return 'key_independent'
    if variable in CONTROL_VARS:
        return 'control'
    return 'supporting'


coverage_by_variable = (
    df.notna()
    .sum()
    .rename('non_missing')
    .to_frame()
    .assign(total=len(df))
    .assign(missing=lambda frame: frame['total'] - frame['non_missing'])
    .assign(missing_rate=lambda frame: frame['missing'] / frame['total'])
)

country_level_presence = df.groupby('country').apply(lambda frame: frame.notna().sum(), include_groups=False)

mechanism_rows = []
for variable in CLEAN_PANEL_COLUMNS:
    if variable in WINSORIZED_SENSITIVITY_COLUMNS or variable in {'country_id', 'country_code', 'country', 'year'}:
        continue
    if variable not in df.columns:
        continue
    missing_flag = df[variable].isna().astype(int)
    if missing_flag.sum() == 0:
        mechanism_rows.append(
            {
                'variable': variable,
                'missing_count': 0,
                'missing_rate': 0.0,
                'model_status': 'not_needed',
                'significant_predictors': '',
                'mechanism_assessment': 'complete',
            }
        )
        continue
    predictors = [predictor for predictor in MECHANISM_PREDICTORS if predictor != variable and predictor in df.columns]
    diagnostic_df = df[['country', 'year', variable, *predictors]].copy()
    diagnostic_df['missing_flag'] = missing_flag
    formula_terms = predictors + ['C(country)', 'year']
    try:
        fit = smf.glm(
            formula='missing_flag ~ ' + ' + '.join(formula_terms),
            data=diagnostic_df,
            family=sm.families.Binomial(),
        ).fit()
        significant_predictors = fit.pvalues.drop(labels=['Intercept'], errors='ignore')
        significant_predictors = significant_predictors[significant_predictors < 0.10]
        mechanism_assessment = 'at_least_MAR' if not significant_predictors.empty else 'MCAR_not_rejected'
        mechanism_rows.append(
            {
                'variable': variable,
                'missing_count': int(missing_flag.sum()),
                'missing_rate': float(missing_flag.mean()),
                'model_status': 'fit',
                'significant_predictors': ', '.join(significant_predictors.index.tolist()),
                'mechanism_assessment': mechanism_assessment,
            }
        )
    except Exception as exc:
        mechanism_rows.append(
            {
                'variable': variable,
                'missing_count': int(missing_flag.sum()),
                'missing_rate': float(missing_flag.mean()),
                'model_status': f'failed: {type(exc).__name__}',
                'significant_predictors': '',
                'mechanism_assessment': 'unresolved_conservative',
            }
        )

missingness_mechanism_diagnostics = pd.DataFrame(mechanism_rows)
missingness_mechanism_diagnostics

/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be ide

,variable,missing_count,missing_rate,model_status,significant_predictors,mechanism_assessment
0,fdi_pct_gdp,0,0.0000,not_needed,,complete
1,broad_money_pct_gdp,21,0.1364,fit,,MCAR_not_rejected
2,trade_pct_gdp,21,0.1364,fit,,MCAR_not_rejected
3,inflation_gdp_deflator_pct,0,0.0000,not_needed,,complete
4,deposit_interest_rate_pct,22,0.1429,fit,,MCAR_not_rejected
5,real_interest_rate_pct,37,0.2403,fit,,MCAR_not_rejected
6,lending_interest_rate_pct,37,0.2403,fit,,MCAR_not_rejected
7,hc_human_capital_index,14,0.0909,fit,,MCAR_not_rejected
8,ln_gdppc,0,0.0000,not_needed,,complete
9,xr_dep_pct,14,0.0909,fit,,MCAR_not_rejected


### 04.1 Missingness And Review Result

Read missingness severity and manual review flags before deciding how variables enter models.


In [7]:
coverage_review = coverage_by_variable.reset_index().rename(columns={'index': 'variable'})
coverage_review['role_group'] = coverage_review['variable'].map(role_group)
display(coverage_review[['variable', 'non_missing', 'missing', 'missing_rate', 'role_group']])
display(missingness_mechanism_diagnostics[['variable', 'missing_count', 'missing_rate', 'model_status', 'mechanism_assessment']])


,variable,non_missing,missing,missing_rate,role_group
0,country_id,154,0,0.0000,supporting
1,country_code,154,0,0.0000,supporting
2,country,154,0,0.0000,supporting
3,year,154,0,0.0000,supporting
4,fdi_pct_gdp,154,0,0.0000,dependent
5,broad_money_pct_gdp,133,21,0.1364,key_independent
6,trade_pct_gdp,133,21,0.1364,control
7,population_total,154,0,0.0000,supporting
8,inflation_gdp_deflator_pct,154,0,0.0000,control
9,official_exchange_rate_lcu_usd,151,3,0.0195,supporting


,variable,missing_count,missing_rate,model_status,mechanism_assessment
0,fdi_pct_gdp,0,0.0000,not_needed,complete
1,broad_money_pct_gdp,21,0.1364,fit,MCAR_not_rejected
2,trade_pct_gdp,21,0.1364,fit,MCAR_not_rejected
3,inflation_gdp_deflator_pct,0,0.0000,not_needed,complete
4,deposit_interest_rate_pct,22,0.1429,fit,MCAR_not_rejected
5,real_interest_rate_pct,37,0.2403,fit,MCAR_not_rejected
6,lending_interest_rate_pct,37,0.2403,fit,MCAR_not_rejected
7,hc_human_capital_index,14,0.0909,fit,MCAR_not_rejected
8,ln_gdppc,0,0.0000,not_needed,complete
9,xr_dep_pct,14,0.0909,fit,MCAR_not_rejected


## 05 Define Missing-Data Handling Rules

Choose conservative handling rules by variable role and missingness structure.

In [8]:
def choose_handling(variable: str, missing_rate: float, structural_missing_countries: int) -> str:
    role = role_group(variable)
    if role == 'dependent':
        return 'observed_only_no_imputation'
    if role == 'key_independent':
        return 'observed_only_no_imputation'
    if role != 'control':
        return 'supporting_only_not_used_for_imputation'
    if variable in NON_INTERPOLATABLE_CONTROLS:
        return 'leave_missing_and_report_limitation'
    if variable == 'xr_dep_pct':
        return 'within_country_interpolate_and_edge_fill_except_structural_2010'
    if missing_rate <= 0.10 and structural_missing_countries == 0:
        return 'within_country_interpolate_then_edge_fill'
    if missing_rate <= 0.30:
        return 'within_country_interpolate_then_edge_fill_keep_structural_country_gaps'
    if missing_rate <= 0.50:
        return 'interpolate_controls_but_treat_as_sensitivity_only'
    return 'leave_missing_and_report_limitation'


handling_rows = []
for variable in CLEAN_PANEL_COLUMNS:
    if variable in WINSORIZED_SENSITIVITY_COLUMNS or variable in {'country_id', 'country_code', 'country', 'year'}:
        continue
    missing_rate = float(df[variable].isna().mean())
    structural_missing_countries = int((country_level_presence[variable] == 0).sum()) if variable in country_level_presence.columns else 0
    severity = (
        '<10%' if missing_rate < 0.10 else
        '10-30%' if missing_rate <= 0.30 else
        '30-50%' if missing_rate <= 0.50 else
        '>50%'
    )
    mechanism = missingness_mechanism_diagnostics.set_index('variable').at[variable, 'mechanism_assessment']
    handling_rows.append(
        {
            'variable': variable,
            'role_group': role_group(variable),
            'missing_rate': missing_rate,
            'severity_band': severity,
            'structural_missing_countries': structural_missing_countries,
            'mechanism_assessment': mechanism,
            'recommended_handling': choose_handling(variable, missing_rate, structural_missing_countries),
        }
    )

missingness_handling_summary = pd.DataFrame(handling_rows).sort_values(['role_group', 'variable']).reset_index(drop=True)
missingness_handling_summary

,variable,role_group,missing_rate,severity_band,structural_missing_countries,mechanism_assessment,recommended_handling
0,hc_human_capital_index,control,0.0909,<10%,1,MCAR_not_rejected,leave_missing_and_report_limitation
1,inflation_gdp_deflator_pct,control,0.0000,<10%,0,complete,within_country_interpolate_then_edge_fill
2,ln_gdppc,control,0.0000,<10%,0,complete,within_country_interpolate_then_edge_fill
3,ln_population_total,control,0.0000,<10%,0,complete,within_country_interpolate_then_edge_fill
4,ln_tourism_arrivals,control,0.2662,10-30%,0,MCAR_not_rejected,within_country_interpolate_then_edge_fill_keep...
5,trade_pct_gdp,control,0.1364,10-30%,1,MCAR_not_rejected,within_country_interpolate_then_edge_fill_keep...
6,xr_dep_pct,control,0.0909,<10%,0,MCAR_not_rejected,within_country_interpolate_and_edge_fill_excep...
7,fdi_pct_gdp,dependent,0.0000,<10%,0,complete,observed_only_no_imputation
8,broad_money_pct_gdp,key_independent,0.1364,10-30%,0,MCAR_not_rejected,observed_only_no_imputation
9,deposit_interest_rate_pct,key_independent,0.1429,10-30%,0,MCAR_not_rejected,observed_only_no_imputation


### 05.1 Missing-Data Handling Result

This is the policy used later: key FDI and monetary-policy variables stay observed-only, while selected controls may be cautiously filled.


In [9]:
handling_view_columns = [
    'variable',
    'role_group',
    'severity_band',
    'structural_missing_countries',
    'mechanism_assessment',
    'recommended_handling',
]
display(missingness_handling_summary[handling_view_columns])


,variable,role_group,severity_band,structural_missing_countries,mechanism_assessment,recommended_handling
0,hc_human_capital_index,control,<10%,1,MCAR_not_rejected,leave_missing_and_report_limitation
1,inflation_gdp_deflator_pct,control,<10%,0,complete,within_country_interpolate_then_edge_fill
2,ln_gdppc,control,<10%,0,complete,within_country_interpolate_then_edge_fill
3,ln_population_total,control,<10%,0,complete,within_country_interpolate_then_edge_fill
4,ln_tourism_arrivals,control,10-30%,0,MCAR_not_rejected,within_country_interpolate_then_edge_fill_keep...
5,trade_pct_gdp,control,10-30%,1,MCAR_not_rejected,within_country_interpolate_then_edge_fill_keep...
6,xr_dep_pct,control,<10%,0,MCAR_not_rejected,within_country_interpolate_and_edge_fill_excep...
7,fdi_pct_gdp,dependent,<10%,0,complete,observed_only_no_imputation
8,broad_money_pct_gdp,key_independent,10-30%,0,MCAR_not_rejected,observed_only_no_imputation
9,deposit_interest_rate_pct,key_independent,10-30%,0,MCAR_not_rejected,observed_only_no_imputation


## 06 Apply Control-Only Imputation

Fill only approved control variables while leaving FDI and monetary-policy variables observed-only.

In [10]:
analysis_df = df.copy()
imputation_rows = []
handling_lookup = missingness_handling_summary.set_index('variable')['recommended_handling'].to_dict()

for variable in CONTROL_VARS:
    if variable not in analysis_df.columns:
        continue
    handling = handling_lookup.get(variable, 'leave_missing_and_report_limitation')
    before_missing = int(analysis_df[variable].isna().sum())
    if handling == 'within_country_interpolate_then_edge_fill' or handling == 'within_country_interpolate_then_edge_fill_keep_structural_country_gaps':
        analysis_df[variable] = analysis_df.groupby('country')[variable].transform(
            lambda series: series.interpolate(method='linear', limit_area='inside').ffill().bfill()
        )
    elif handling == 'within_country_interpolate_and_edge_fill_except_structural_2010':
        structural_mask = analysis_df['year'].eq(2010)
        work = analysis_df[variable].copy()
        imputed = analysis_df.groupby('country')[variable].transform(
            lambda series: series.interpolate(method='linear', limit_area='inside').ffill().bfill()
        )
        work.loc[~structural_mask] = imputed.loc[~structural_mask]
        analysis_df[variable] = work
    elif handling == 'interpolate_controls_but_treat_as_sensitivity_only':
        analysis_df[variable] = analysis_df.groupby('country')[variable].transform(
            lambda series: series.interpolate(method='linear', limit_area='inside').ffill().bfill()
        )
    after_missing = int(analysis_df[variable].isna().sum())
    imputation_rows.append(
        {
            'variable': variable,
            'handling_applied': handling,
            'missing_before': before_missing,
            'missing_after': after_missing,
            'filled_values': before_missing - after_missing,
        }
    )

control_imputation_log = pd.DataFrame(imputation_rows)

winsorization_rows = []
for source_col in ['fdi_pct_gdp', 'xr_dep_pct']:
    target_col = f'{source_col}_winsorized'
    low, high = analysis_df[source_col].quantile([0.01, 0.99])
    analysis_df[target_col] = analysis_df[source_col].clip(lower=low, upper=high)
    winsorization_rows.append(
        {
            'source_variable': source_col,
            'sensitivity_variable': target_col,
            'lower_bound_p01': float(low),
            'upper_bound_p99': float(high),
            'values_clipped': int(analysis_df[source_col].ne(analysis_df[target_col]).sum()),
            'use_case': 'robustness_only_do_not_replace_main_observed_values',
        }
    )

winsorization_thresholds = pd.DataFrame(winsorization_rows)
control_imputation_log

,variable,handling_applied,missing_before,missing_after,filled_values
0,trade_pct_gdp,within_country_interpolate_then_edge_fill_keep...,21,14,7
1,inflation_gdp_deflator_pct,within_country_interpolate_then_edge_fill,0,0,0
2,ln_gdppc,within_country_interpolate_then_edge_fill,0,0,0
3,xr_dep_pct,within_country_interpolate_and_edge_fill_excep...,14,11,3
4,ln_population_total,within_country_interpolate_then_edge_fill,0,0,0
5,ln_tourism_arrivals,within_country_interpolate_then_edge_fill_keep...,41,0,41
6,hc_human_capital_index,leave_missing_and_report_limitation,14,14,0


### 06.1 Control Imputation Result

Show only variables touched by the control-imputation policy.


In [11]:
display(control_imputation_log)
display(analysis_df.head())


,variable,handling_applied,missing_before,missing_after,filled_values
0,trade_pct_gdp,within_country_interpolate_then_edge_fill_keep...,21,14,7
1,inflation_gdp_deflator_pct,within_country_interpolate_then_edge_fill,0,0,0
2,ln_gdppc,within_country_interpolate_then_edge_fill,0,0,0
3,xr_dep_pct,within_country_interpolate_and_edge_fill_excep...,14,11,3
4,ln_population_total,within_country_interpolate_then_edge_fill,0,0,0
5,ln_tourism_arrivals,within_country_interpolate_then_edge_fill_keep...,41,0,41
6,hc_human_capital_index,leave_missing_and_report_limitation,14,14,0


,country_id,country_code,country,year,fdi_pct_gdp,broad_money_pct_gdp,trade_pct_gdp,population_total,inflation_gdp_deflator_pct,official_exchange_rate_lcu_usd,gdppc_current_usd,deposit_interest_rate_pct,tourism_arrivals,hc_human_capital_index,ln_gdppc,xr_dep_pct,lending_interest_rate_pct,real_interest_rate_pct,ln_gdppc_recomputed,xr_dep_pct_recomputed,ln_gdppc_diff,xr_dep_pct_diff,ln_population_total,ln_tourism_arrivals,fdi_pct_gdp_winsorized,xr_dep_pct_winsorized
0,1,BRN,Brunei Darussalam,2010,3.5071,67.2720,95.3715,392332,4.9801,1.3635,"34,937.5555",0.4705,NaN,2.6970,10.4613,NaN,5.5000,0.4952,10.4613,NaN,-0.0000,NaN,12.8799,15.2641,3.5071,NaN
1,1,BRN,Brunei Darussalam,2011,3.7311,59.3805,99.5379,399389,20.1808,1.2579,"46,382.8274",0.3957,NaN,2.7215,10.7447,-8.0608,5.5000,-12.2156,10.7447,-8.0608,-0.0000,0.0000,12.8977,15.2641,3.7311,-8.0608
2,1,BRN,Brunei Darussalam,2012,4.5406,58.6565,105.6409,405557,1.2203,1.2496,"46,968.5971",0.2315,NaN,2.7325,10.7572,-0.6657,5.5000,4.2282,10.7572,-0.6657,0.0000,0.0000,12.9130,15.2641,4.5406,-0.6657
3,1,BRN,Brunei Darussalam,2013,4.2867,62.5754,110.9396,411202,-2.8235,1.2512,"44,003.0644",0.2843,NaN,2.7436,10.6920,0.1279,5.5000,8.5653,10.6920,0.1279,-0.0000,0.0000,12.9268,15.2641,4.2867,0.1279
4,1,BRN,Brunei Darussalam,2014,3.3566,67.4986,102.4210,416750,-1.8460,1.2670,"41,026.5084",0.3000,NaN,2.7549,10.6220,1.2608,5.5000,7.4842,10.6220,1.2608,0.0000,0.0000,12.9402,15.2641,3.3566,1.2608


## 07 Build Model-Ready Audit Tables

Create transformation, coverage, review, and specification-sample tables.

### 07.1 Transformation Audit

Record which variables are already transformed, which are created in the notebook, and which should be preferred for modeling.


In [12]:
transformation_audit = pd.DataFrame(
    [
        {
            'source_variable': 'gdppc_current_usd',
            'transformed_variable': 'ln_gdppc',
            'transformation': 'natural_log',
            'status': 'already_in_dataset',
            'preferred_for_modeling': True,
        },
        {
            'source_variable': 'official_exchange_rate_lcu_usd',
            'transformed_variable': 'xr_dep_pct',
            'transformation': 'country_log_difference_x100',
            'status': 'already_in_dataset_then_missingness_workflow_applied',
            'preferred_for_modeling': True,
        },
        {
            'source_variable': 'population_total',
            'transformed_variable': 'ln_population_total',
            'transformation': 'natural_log',
            'status': 'created_in_notebook',
            'preferred_for_modeling': True,
        },
        {
            'source_variable': 'tourism_arrivals',
            'transformed_variable': 'ln_tourism_arrivals',
            'transformation': 'natural_log',
            'status': 'created_in_notebook_then_missingness_workflow_applied',
            'preferred_for_modeling': True,
        },
        {
            'source_variable': 'data/raw/*.xlsx',
            'transformed_variable': 'real_interest_rate_pct',
            'transformation': 'country_year_merge_from_available_wdi_series',
            'status': 'supplemented_from_raw_directory',
            'preferred_for_modeling': False,
        },
        {
            'source_variable': 'data/raw/*.xlsx',
            'transformed_variable': 'lending_interest_rate_pct',
            'transformation': 'country_year_merge_from_available_wdi_series',
            'status': 'supplemented_from_raw_directory',
            'preferred_for_modeling': False,
        },
        {
            'source_variable': 'fdi_pct_gdp',
            'transformed_variable': 'fdi_pct_gdp_winsorized',
            'transformation': 'clip_to_1st_99th_percentiles_for_sensitivity_only',
            'status': 'created_in_notebook_after_review_flagging',
            'preferred_for_modeling': False,
        },
        {
            'source_variable': 'xr_dep_pct',
            'transformed_variable': 'xr_dep_pct_winsorized',
            'transformation': 'clip_to_1st_99th_percentiles_for_sensitivity_only',
            'status': 'created_in_notebook_after_review_flagging',
            'preferred_for_modeling': False,
        },
    ]
)
transformation_audit['non_missing_source'] = transformation_audit['source_variable'].map(analysis_df.notna().sum())
transformation_audit['non_missing_transformed'] = transformation_audit['transformed_variable'].map(analysis_df.notna().sum())

### 07.1 Result

Transformation audit used by the later modeling notebooks.


In [13]:
display(transformation_audit)


,source_variable,transformed_variable,transformation,status,preferred_for_modeling,non_missing_source,non_missing_transformed
0,gdppc_current_usd,ln_gdppc,natural_log,already_in_dataset,True,154.0000,154
1,official_exchange_rate_lcu_usd,xr_dep_pct,country_log_difference_x100,already_in_dataset_then_missingness_workflow_a...,True,151.0000,143
2,population_total,ln_population_total,natural_log,created_in_notebook,True,154.0000,154
3,tourism_arrivals,ln_tourism_arrivals,natural_log,created_in_notebook_then_missingness_workflow_...,True,113.0000,154
4,data/raw/*.xlsx,real_interest_rate_pct,country_year_merge_from_available_wdi_series,supplemented_from_raw_directory,False,NaN,117
5,data/raw/*.xlsx,lending_interest_rate_pct,country_year_merge_from_available_wdi_series,supplemented_from_raw_directory,False,NaN,117
6,fdi_pct_gdp,fdi_pct_gdp_winsorized,clip_to_1st_99th_percentiles_for_sensitivity_only,created_in_notebook_after_review_flagging,False,154.0000,154
7,xr_dep_pct,xr_dep_pct_winsorized,clip_to_1st_99th_percentiles_for_sensitivity_only,created_in_notebook_after_review_flagging,False,143.0000,143


### 07.2 Coverage, Review Flags, And Sensitivity Columns

Check country coverage, flag extreme values for interpretation, and create winsorized sensitivity variables without replacing the observed values used in the main panel.

In [14]:
coverage_by_country = (
    analysis_df.groupby('country')[
        [
            'fdi_pct_gdp',
            'broad_money_pct_gdp',
            'deposit_interest_rate_pct',
            'real_interest_rate_pct',
            'lending_interest_rate_pct',
            'trade_pct_gdp',
            'inflation_gdp_deflator_pct',
            'ln_gdppc',
            'xr_dep_pct',
            'ln_tourism_arrivals',
            'ln_population_total',
            'hc_human_capital_index',
        ]
    ]
    .apply(lambda frame: frame.notna().sum())
    .sort_index()
)

panel_grid = (
    analysis_df.assign(observed=1)
    .pivot(index='country', columns='year', values='observed')
    .fillna(0)
    .astype(int)
    .sort_index(axis=0)
    .sort_index(axis=1)
)

review_flag_frames = [
    analysis_df.loc[analysis_df['xr_dep_pct'].abs() > 100, ['country', 'year']].assign(
        flag='xr_dep_pct_extreme_keep_and_review',
        value=lambda frame: analysis_df.loc[frame.index, 'xr_dep_pct'],
        note='Observed value retained; use winsorized version only for sensitivity checks.',
    ),
    analysis_df.loc[analysis_df['inflation_gdp_deflator_pct'].abs() > 30, ['country', 'year']].assign(
        flag='inflation_extreme_keep_and_review',
        value=lambda frame: analysis_df.loc[frame.index, 'inflation_gdp_deflator_pct'],
        note='Observed value retained for main analysis.',
    ),
    analysis_df.loc[analysis_df['fdi_pct_gdp'].abs() > 50, ['country', 'year']].assign(
        flag='fdi_truly_extreme_keep_and_review',
        value=lambda frame: analysis_df.loc[frame.index, 'fdi_pct_gdp'],
        note='Observed value retained; inspect country-year leverage before interpreting pooled models.',
    ),
    analysis_df.loc[
        analysis_df['country'].eq('Singapore') & analysis_df['fdi_pct_gdp'].abs().gt(20),
        ['country', 'year'],
    ].assign(
        flag='singapore_fdi_gt_20_structural_expected',
        value=lambda frame: analysis_df.loc[frame.index, 'fdi_pct_gdp'],
        note='High Singapore FDI-to-GDP is economically expected; keep and rely on country FE/sensitivity checks.',
    ),
    analysis_df.loc[
        ~analysis_df['country'].eq('Singapore') & analysis_df['fdi_pct_gdp'].abs().gt(20),
        ['country', 'year'],
    ].assign(
        flag='non_singapore_fdi_gt_20_keep_and_review',
        value=lambda frame: analysis_df.loc[frame.index, 'fdi_pct_gdp'],
        note='Observed value retained; review country-year influence.',
    ),
    analysis_df.loc[analysis_df['trade_pct_gdp'] > 300, ['country', 'year']].assign(
        flag='trade_high_open_economy_review',
        value=lambda frame: analysis_df.loc[frame.index, 'trade_pct_gdp'],
        note='High openness can be structural for entrepot economies; retain and interpret with country effects.',
    ),
]
review_flags = (
    pd.concat(review_flag_frames, ignore_index=True)
    .loc[:, ['country', 'year', 'flag', 'value', 'note']]
    .sort_values(['flag', 'country', 'year'])
    .reset_index(drop=True)
)

### 07.2 Result

Country coverage, review flags, and sensitivity-only winsorization thresholds are shown immediately so gaps and influential values are visible before modeling.

In [15]:
display(coverage_by_country)
display(review_flags.head(40))
display(winsorization_thresholds)

,fdi_pct_gdp,broad_money_pct_gdp,deposit_interest_rate_pct,real_interest_rate_pct,lending_interest_rate_pct,trade_pct_gdp,inflation_gdp_deflator_pct,ln_gdppc,xr_dep_pct,ln_tourism_arrivals,ln_population_total,hc_human_capital_index
country,,,,,,,,,,,,
Brunei Darussalam,14,14,14,14,14,14,14,14,13,14,14,14
Cambodia,14,14,14,0,0,14,14,14,13,14,14,14
Indonesia,14,14,14,14,14,14,14,14,13,14,14,14
Lao PDR,14,1,1,1,1,14,14,14,13,14,14,14
Malaysia,14,14,14,14,14,14,14,14,13,14,14,14
Myanmar,14,11,11,11,11,0,14,14,13,14,14,14
Philippines,14,13,10,10,10,14,14,14,13,14,14,14
Singapore,14,11,12,12,12,14,14,14,13,14,14,14
Thailand,14,14,14,14,14,14,14,14,13,14,14,14


,country,year,flag,value,note
0,Timor-Leste,2021,inflation_extreme_keep_and_review,59.0797,Observed value retained for main analysis.
1,Viet Nam,2010,inflation_extreme_keep_and_review,42.3033,Observed value retained for main analysis.
2,Singapore,2010,singapore_fdi_gt_20_structural_expected,23.0695,High Singapore FDI-to-GDP is economically expe...
3,Singapore,2013,singapore_fdi_gt_20_structural_expected,20.9345,High Singapore FDI-to-GDP is economically expe...
4,Singapore,2014,singapore_fdi_gt_20_structural_expected,21.8185,High Singapore FDI-to-GDP is economically expe...
5,Singapore,2015,singapore_fdi_gt_20_structural_expected,22.6542,High Singapore FDI-to-GDP is economically expe...
6,Singapore,2016,singapore_fdi_gt_20_structural_expected,20.3766,High Singapore FDI-to-GDP is economically expe...
7,Singapore,2017,singapore_fdi_gt_20_structural_expected,29.6706,High Singapore FDI-to-GDP is economically expe...
8,Singapore,2018,singapore_fdi_gt_20_structural_expected,22.8377,High Singapore FDI-to-GDP is economically expe...
9,Singapore,2019,singapore_fdi_gt_20_structural_expected,28.2257,High Singapore FDI-to-GDP is economically expe...


,source_variable,sensitivity_variable,lower_bound_p01,upper_bound_p99,values_clipped,use_case
0,fdi_pct_gdp,fdi_pct_gdp_winsorized,-7.1580,28.9542,4,robustness_only_do_not_replace_main_observed_v...
1,xr_dep_pct,xr_dep_pct_winsorized,-9.4306,37.3643,13,robustness_only_do_not_replace_main_observed_v...


### 07.3 Specification Samples

Estimate how much each candidate variable set shrinks the usable panel before running econometric models. The saturated specification is labeled diagnostic-only because it intentionally combines variables known to create collinearity pressure.

In [16]:
SPECIFICATIONS = {
    'baseline_theory_fit': ['fdi_pct_gdp', 'broad_money_pct_gdp', 'deposit_interest_rate_pct', 'trade_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc'],
    'baseline_with_xr': ['fdi_pct_gdp', 'broad_money_pct_gdp', 'deposit_interest_rate_pct', 'trade_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc', 'xr_dep_pct'],
    'liquidity_broad_sample': ['fdi_pct_gdp', 'broad_money_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc'],
    'spec_real_rate': ['fdi_pct_gdp', 'real_interest_rate_pct', 'trade_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc', 'xr_dep_pct'],
    'spec_deposit_rate': ['fdi_pct_gdp', 'deposit_interest_rate_pct', 'trade_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc', 'xr_dep_pct'],
    'spec_broad_money_only': ['fdi_pct_gdp', 'broad_money_pct_gdp', 'trade_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc', 'xr_dep_pct'],
    'real_interest_robustness': ['fdi_pct_gdp', 'broad_money_pct_gdp', 'real_interest_rate_pct', 'trade_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc', 'xr_dep_pct'],
    'lending_rate_robustness': ['fdi_pct_gdp', 'broad_money_pct_gdp', 'lending_interest_rate_pct', 'trade_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc', 'xr_dep_pct'],
    'full_saturated_diagnostic_only': ['fdi_pct_gdp', 'broad_money_pct_gdp', 'deposit_interest_rate_pct', 'trade_pct_gdp', 'inflation_gdp_deflator_pct', 'ln_gdppc', 'xr_dep_pct', 'ln_population_total', 'ln_tourism_arrivals', 'hc_human_capital_index'],
}

SPECIFICATION_ROLES = {
    'baseline_theory_fit': 'main_baseline',
    'baseline_with_xr': 'main_baseline_extension',
    'liquidity_broad_sample': 'liquidity_sensitivity',
    'spec_real_rate': 'one_proxy_per_channel_sensitivity',
    'spec_deposit_rate': 'one_proxy_per_channel_sensitivity',
    'spec_broad_money_only': 'one_proxy_per_channel_sensitivity',
    'real_interest_robustness': 'robustness_collinearity_check',
    'lending_rate_robustness': 'robustness_collinearity_check',
    'full_saturated_diagnostic_only': 'diagnostic_only_not_preferred',
}

sample_rows = []
sample_members = []
for spec_name, columns in SPECIFICATIONS.items():
    mask = analysis_df[columns].notna().all(axis=1)
    subset = analysis_df.loc[mask, ['country', 'year']].copy()
    sample_rows.append(
        {
            'specification': spec_name,
            'specification_role': SPECIFICATION_ROLES[spec_name],
            'rows_used': int(mask.sum()),
            'rows_dropped': int((~mask).sum()),
            'countries_used': int(subset['country'].nunique()),
            'years_used': int(subset['year'].nunique()) if not subset.empty else 0,
            'variables': ', '.join(columns),
        }
    )
    sample_members.append(subset.assign(specification=spec_name, in_sample=1))

spec_sample_summary = pd.DataFrame(sample_rows)
spec_sample_membership = pd.concat(sample_members, ignore_index=True)
spec_sample_country_counts = (
    spec_sample_membership.groupby(['specification', 'country'])['in_sample']
    .sum()
    .rename('rows_used')
    .reset_index()
)

clean_panel = analysis_df[CLEAN_PANEL_COLUMNS].copy()

### 07.3 Result

Candidate specification sample sizes before econometric estimation.


In [17]:
display(spec_sample_summary)
display(spec_sample_country_counts.head(40))


,specification,specification_role,rows_used,rows_dropped,countries_used,years_used,variables
0,baseline_theory_fit,main_baseline,119,35,10,14,"fdi_pct_gdp, broad_money_pct_gdp, deposit_inte..."
1,baseline_with_xr,main_baseline_extension,109,45,9,13,"fdi_pct_gdp, broad_money_pct_gdp, deposit_inte..."
2,liquidity_broad_sample,liquidity_sensitivity,133,21,11,14,"fdi_pct_gdp, broad_money_pct_gdp, inflation_gd..."
3,spec_real_rate,one_proxy_per_channel_sensitivity,98,56,8,13,"fdi_pct_gdp, real_interest_rate_pct, trade_pct..."
4,spec_deposit_rate,one_proxy_per_channel_sensitivity,111,43,9,13,"fdi_pct_gdp, deposit_interest_rate_pct, trade_..."
5,spec_broad_money_only,one_proxy_per_channel_sensitivity,112,42,9,13,"fdi_pct_gdp, broad_money_pct_gdp, trade_pct_gd..."
6,real_interest_robustness,robustness_collinearity_check,96,58,8,13,"fdi_pct_gdp, broad_money_pct_gdp, real_interes..."
7,lending_rate_robustness,robustness_collinearity_check,96,58,8,13,"fdi_pct_gdp, broad_money_pct_gdp, lending_inte..."
8,full_saturated_diagnostic_only,diagnostic_only_not_preferred,96,58,8,13,"fdi_pct_gdp, broad_money_pct_gdp, deposit_inte..."


,specification,country,rows_used
0,baseline_theory_fit,Brunei Darussalam,14
1,baseline_theory_fit,Cambodia,14
2,baseline_theory_fit,Indonesia,14
3,baseline_theory_fit,Lao PDR,1
4,baseline_theory_fit,Malaysia,14
5,baseline_theory_fit,Philippines,10
6,baseline_theory_fit,Singapore,11
7,baseline_theory_fit,Thailand,14
8,baseline_theory_fit,Timor-Leste,14
9,baseline_theory_fit,Viet Nam,13


### 07.4 Specification VIF Audit

Compute a formal multicollinearity diagnostic for each candidate specification. This is an audit table, not a model result, and it should guide whether a specification belongs in the main table or only in sensitivity/appendix material.

In [18]:
vif_rows = []
for spec_name, columns in SPECIFICATIONS.items():
    predictors = [column for column in columns if column != DEP_VAR]
    spec_df = analysis_df[predictors].dropna()
    if len(spec_df) < len(predictors) + 5:
        vif_rows.append(
            {
                'specification': spec_name,
                'variable': None,
                'vif': np.nan,
                'rows_used': int(len(spec_df)),
                'multicollinearity_flag': 'insufficient_rows_for_vif',
            }
        )
        continue
    exog = spec_df.astype(float)
    for index, variable in enumerate(predictors):
        try:
            vif_value = float(variance_inflation_factor(exog.values, index))
        except Exception:
            vif_value = np.nan
        vif_rows.append(
            {
                'specification': spec_name,
                'variable': variable,
                'vif': vif_value,
                'rows_used': int(len(spec_df)),
                'multicollinearity_flag': (
                    'severe' if pd.notna(vif_value) and vif_value > 10 else
                    'moderate' if pd.notna(vif_value) and vif_value > 5 else
                    'acceptable' if pd.notna(vif_value) else
                    'not_computed'
                ),
            }
        )

spec_vif_table = pd.DataFrame(vif_rows)
spec_vif_summary = (
    spec_vif_table.dropna(subset=['variable'])
    .groupby('specification')
    .agg(
        max_vif=('vif', 'max'),
        severe_vif_variables=('multicollinearity_flag', lambda series: int(series.eq('severe').sum())),
        moderate_vif_variables=('multicollinearity_flag', lambda series: int(series.eq('moderate').sum())),
    )
    .reset_index()
    .merge(spec_sample_summary[['specification', 'specification_role', 'rows_used', 'countries_used']], on='specification', how='left')
    .sort_values(['severe_vif_variables', 'max_vif'], ascending=[False, False])
    .reset_index(drop=True)
)

### 07.4 Result

Specifications with max VIF above 10 should be treated as high-collinearity diagnostics unless there is a strong theoretical reason to keep them.

In [19]:
display(spec_vif_summary)
display(spec_vif_table.sort_values(['specification', 'vif'], ascending=[True, False]).head(60))

,specification,max_vif,severe_vif_variables,moderate_vif_variables,specification_role,rows_used,countries_used
0,full_saturated_diagnostic_only,535.0569,5,1,diagnostic_only_not_preferred,96,8
1,lending_rate_robustness,22.6259,2,2,robustness_collinearity_check,96,8
2,real_interest_robustness,21.2692,2,3,robustness_collinearity_check,96,8
3,baseline_with_xr,11.1325,2,1,main_baseline_extension,109,9
4,baseline_theory_fit,10.8316,2,1,main_baseline,119,10
5,spec_broad_money_only,11.1114,1,2,one_proxy_per_channel_sensitivity,112,9
6,spec_real_rate,10.0474,1,2,one_proxy_per_channel_sensitivity,98,8
7,liquidity_broad_sample,8.6419,0,2,liquidity_sensitivity,133,11
8,spec_deposit_rate,7.3115,0,2,one_proxy_per_channel_sensitivity,111,9


,specification,variable,vif,rows_used,multicollinearity_flag
0,baseline_theory_fit,broad_money_pct_gdp,10.8316,119,severe
4,baseline_theory_fit,ln_gdppc,10.6445,119,severe
2,baseline_theory_fit,trade_pct_gdp,6.8839,119,moderate
1,baseline_theory_fit,deposit_interest_rate_pct,2.1631,119,acceptable
3,baseline_theory_fit,inflation_gdp_deflator_pct,1.3119,119,acceptable
5,baseline_with_xr,broad_money_pct_gdp,11.1325,109,severe
9,baseline_with_xr,ln_gdppc,10.9016,109,severe
7,baseline_with_xr,trade_pct_gdp,7.0120,109,moderate
6,baseline_with_xr,deposit_interest_rate_pct,2.2794,109,acceptable
10,baseline_with_xr,xr_dep_pct,1.2593,109,acceptable


### 07.4 Clean Panel Preview

The processed panel remains the full country-year foundation. Model-specific complete-case samples are formed later.


In [20]:
clean_panel.head()


,country_id,country_code,country,year,fdi_pct_gdp,fdi_pct_gdp_winsorized,broad_money_pct_gdp,trade_pct_gdp,inflation_gdp_deflator_pct,deposit_interest_rate_pct,real_interest_rate_pct,lending_interest_rate_pct,hc_human_capital_index,ln_gdppc,xr_dep_pct,xr_dep_pct_winsorized,ln_population_total,ln_tourism_arrivals
0,1,BRN,Brunei Darussalam,2010,3.5071,3.5071,67.2720,95.3715,4.9801,0.4705,0.4952,5.5000,2.6970,10.4613,NaN,NaN,12.8799,15.2641
1,1,BRN,Brunei Darussalam,2011,3.7311,3.7311,59.3805,99.5379,20.1808,0.3957,-12.2156,5.5000,2.7215,10.7447,-8.0608,-8.0608,12.8977,15.2641
2,1,BRN,Brunei Darussalam,2012,4.5406,4.5406,58.6565,105.6409,1.2203,0.2315,4.2282,5.5000,2.7325,10.7572,-0.6657,-0.6657,12.9130,15.2641
3,1,BRN,Brunei Darussalam,2013,4.2867,4.2867,62.5754,110.9396,-2.8235,0.2843,8.5653,5.5000,2.7436,10.6920,0.1279,0.1279,12.9268,15.2641
4,1,BRN,Brunei Darussalam,2014,3.3566,3.3566,67.4986,102.4210,-1.8460,0.3000,7.4842,5.5000,2.7549,10.6220,1.2608,1.2608,12.9402,15.2641


## 08 Panel Balance And Sample Reliability Audit

Check whether complete-case estimation leaves uneven country time windows, internal year gaps, or too few observations per country before modeling.


### 08.1 Country-Year Window Helper

Use a common helper to distinguish dropped countries, short windows, and internal year gaps.


In [21]:
def summarize_country_year_window(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    rows = []
    for group_key, group in frame.groupby(group_columns, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        years = sorted(group['year'].dropna().astype(int).unique())
        row = dict(zip(group_columns, group_key))
        if years:
            full_window = set(range(min(years), max(years) + 1))
            missing_inside = sorted(full_window - set(years))
            row.update(
                {
                    'usable_years': len(years),
                    'first_usable_year': min(years),
                    'last_usable_year': max(years),
                    'window_length_years': max(years) - min(years) + 1,
                    'internal_missing_years': len(missing_inside),
                    'internal_missing_year_list': ', '.join(map(str, missing_inside)) if missing_inside else '',
                    'is_contiguous_window': len(missing_inside) == 0,
                }
            )
        else:
            row.update(
                {
                    'usable_years': 0,
                    'first_usable_year': pd.NA,
                    'last_usable_year': pd.NA,
                    'window_length_years': 0,
                    'internal_missing_years': 0,
                    'internal_missing_year_list': '',
                    'is_contiguous_window': False,
                }
            )
        rows.append(row)
    return pd.DataFrame(rows)


### 08.2 Baseline Panel Balance

Confirm whether the processed panel itself is balanced before checking variable-driven sample loss.


In [22]:
all_country_years = analysis_df[['country', 'year']].drop_duplicates()
base_panel_balance_by_country = summarize_country_year_window(all_country_years, ['country'])


### 08.2 Result

The processed panel foundation before variable-driven complete-case filtering.


In [23]:
display(base_panel_balance_by_country)


,country,usable_years,first_usable_year,last_usable_year,window_length_years,internal_missing_years,internal_missing_year_list,is_contiguous_window
0,Brunei Darussalam,14,2010,2023,14,0,,True
1,Cambodia,14,2010,2023,14,0,,True
2,Indonesia,14,2010,2023,14,0,,True
3,Lao PDR,14,2010,2023,14,0,,True
4,Malaysia,14,2010,2023,14,0,,True
5,Myanmar,14,2010,2023,14,0,,True
6,Philippines,14,2010,2023,14,0,,True
7,Singapore,14,2010,2023,14,0,,True
8,Thailand,14,2010,2023,14,0,,True
9,Timor-Leste,14,2010,2023,14,0,,True


### 08.3 Specification-Level Panel Balance

Summarize how complete-case rules change the country-year window for each candidate specification.


In [24]:
spec_country_frames = []
for spec_name, columns in SPECIFICATIONS.items():
    mask = analysis_df[columns].notna().all(axis=1)
    spec_country_frames.append(
        analysis_df.loc[mask, ['country', 'year']].assign(specification=spec_name)
    )

spec_sample_country_years = pd.concat(spec_country_frames, ignore_index=True)
spec_panel_balance_by_country = summarize_country_year_window(
    spec_sample_country_years,
    ['specification', 'country'],
).sort_values(['specification', 'country']).reset_index(drop=True)

spec_panel_balance_summary = (
    spec_panel_balance_by_country.groupby('specification')
    .agg(
        countries_used=('country', 'nunique'),
        total_rows=('usable_years', 'sum'),
        min_country_years=('usable_years', 'min'),
        median_country_years=('usable_years', 'median'),
        max_country_years=('usable_years', 'max'),
        countries_with_lt_8_years=('usable_years', lambda series: int((series < 8).sum())),
        countries_with_internal_gaps=('internal_missing_years', lambda series: int((series > 0).sum())),
    )
    .reset_index()
)
spec_panel_balance_summary['panel_balance_warning'] = np.select(
    [
        spec_panel_balance_summary['countries_used'].lt(8),
        spec_panel_balance_summary['countries_with_lt_8_years'].gt(0),
        spec_panel_balance_summary['countries_with_internal_gaps'].gt(0),
    ],
    [
        'high_risk_fewer_than_8_countries',
        'review_country_with_short_time_series',
        'review_internal_year_gaps',
    ],
    default='acceptable_for_unbalanced_panel_with_caveat',
)


### 08.3 Result

Specification-level balance flags are available here, not only in the final export.


In [25]:
display(spec_panel_balance_summary)
display(spec_panel_balance_by_country.head(40))


,specification,countries_used,total_rows,min_country_years,median_country_years,max_country_years,countries_with_lt_8_years,countries_with_internal_gaps,panel_balance_warning
0,baseline_theory_fit,10,119,1,14.0000,14,1,0,review_country_with_short_time_series
1,baseline_with_xr,9,109,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
2,full_saturated_diagnostic_only,8,96,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
3,lending_rate_robustness,8,96,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
4,liquidity_broad_sample,11,133,1,14.0000,14,1,0,review_country_with_short_time_series
5,real_interest_robustness,8,96,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
6,spec_broad_money_only,9,112,10,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
7,spec_deposit_rate,9,111,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
8,spec_real_rate,8,98,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat


,specification,country,usable_years,first_usable_year,last_usable_year,window_length_years,internal_missing_years,internal_missing_year_list,is_contiguous_window
0,baseline_theory_fit,Brunei Darussalam,14,2010,2023,14,0,,True
1,baseline_theory_fit,Cambodia,14,2010,2023,14,0,,True
2,baseline_theory_fit,Indonesia,14,2010,2023,14,0,,True
3,baseline_theory_fit,Lao PDR,1,2010,2010,1,0,,True
4,baseline_theory_fit,Malaysia,14,2010,2023,14,0,,True
5,baseline_theory_fit,Philippines,10,2010,2019,10,0,,True
6,baseline_theory_fit,Singapore,11,2010,2020,11,0,,True
7,baseline_theory_fit,Thailand,14,2010,2023,14,0,,True
8,baseline_theory_fit,Timor-Leste,14,2010,2023,14,0,,True
9,baseline_theory_fit,Viet Nam,13,2010,2022,13,0,,True


### 08.4 Variable Coverage Windows

Identify variables whose missingness removes countries or shortens time coverage.


In [26]:
variable_country_coverage_rows = []
model_variables_for_coverage = sorted({column for columns in SPECIFICATIONS.values() for column in columns})
for variable in model_variables_for_coverage:
    if variable not in analysis_df.columns:
        continue
    observed = analysis_df.loc[analysis_df[variable].notna(), ['country', 'year']].copy()
    if observed.empty:
        continue
    coverage = summarize_country_year_window(observed, ['country'])
    coverage['variable'] = variable
    variable_country_coverage_rows.append(coverage)

variable_country_coverage_windows = pd.concat(variable_country_coverage_rows, ignore_index=True)[
    [
        'variable',
        'country',
        'usable_years',
        'first_usable_year',
        'last_usable_year',
        'window_length_years',
        'internal_missing_years',
        'internal_missing_year_list',
        'is_contiguous_window',
    ]
].sort_values(['variable', 'country']).reset_index(drop=True)


### 08.4 Result

Variable-specific country windows identify which variables shrink the model sample.


In [27]:
display(variable_country_coverage_windows.head(60))


,variable,country,usable_years,first_usable_year,last_usable_year,window_length_years,internal_missing_years,internal_missing_year_list,is_contiguous_window
0,broad_money_pct_gdp,Brunei Darussalam,14,2010,2023,14,0,,True
1,broad_money_pct_gdp,Cambodia,14,2010,2023,14,0,,True
2,broad_money_pct_gdp,Indonesia,14,2010,2023,14,0,,True
3,broad_money_pct_gdp,Lao PDR,1,2010,2010,1,0,,True
4,broad_money_pct_gdp,Malaysia,14,2010,2023,14,0,,True
5,broad_money_pct_gdp,Myanmar,11,2010,2020,11,0,,True
6,broad_money_pct_gdp,Philippines,13,2010,2022,13,0,,True
7,broad_money_pct_gdp,Singapore,11,2010,2020,11,0,,True
8,broad_money_pct_gdp,Thailand,14,2010,2023,14,0,,True
9,broad_money_pct_gdp,Timor-Leste,14,2010,2023,14,0,,True


### 08.5 Sample Reliability Note

Translate the balance audit into modeling caveats that should appear in the thesis.


In [28]:
short_panel_threshold_years = 8
panel_balance_method_note = pd.DataFrame(
    [
        {
            'issue': 'complete_case_unbalanced_panel',
            'current_status': 'complete-case samples differ across model specifications',
            'risk': 'coefficients may reflect country/year coverage differences as well as economic effects',
            'recommended_response': 'report country-year window audits, keep main model parsimonious, and use common-sample sensitivity checks',
        },
        {
            'issue': 'short_country_series',
            'current_status': f'flag country-spec samples with fewer than {short_panel_threshold_years} usable years',
            'risk': 'country fixed effects become weakly informed for countries with very short usable windows',
            'recommended_response': 'avoid making country-specific claims and test robustness to minimum-year thresholds',
        },
        {
            'issue': 'control_imputation_scope',
            'current_status': 'only controls are interpolated; FDI and monetary-policy variables remain observed-only',
            'risk': 'key-variable missingness still drives sample loss',
            'recommended_response': 'make imputation flags visible and avoid imputing core monetary-policy variables without a separate sensitivity design',
        },
    ]
)


### 08.6 Panel Balance Review

Notebook-facing views stay compact; full country-level tables are exported in the final section.


In [29]:
display(spec_panel_balance_summary)
display(spec_panel_balance_by_country.head(30))
display(variable_country_coverage_windows.head(30))
display(panel_balance_method_note)


,specification,countries_used,total_rows,min_country_years,median_country_years,max_country_years,countries_with_lt_8_years,countries_with_internal_gaps,panel_balance_warning
0,baseline_theory_fit,10,119,1,14.0000,14,1,0,review_country_with_short_time_series
1,baseline_with_xr,9,109,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
2,full_saturated_diagnostic_only,8,96,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
3,lending_rate_robustness,8,96,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
4,liquidity_broad_sample,11,133,1,14.0000,14,1,0,review_country_with_short_time_series
5,real_interest_robustness,8,96,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
6,spec_broad_money_only,9,112,10,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
7,spec_deposit_rate,9,111,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat
8,spec_real_rate,8,98,9,13.0000,13,0,0,acceptable_for_unbalanced_panel_with_caveat


,specification,country,usable_years,first_usable_year,last_usable_year,window_length_years,internal_missing_years,internal_missing_year_list,is_contiguous_window
0,baseline_theory_fit,Brunei Darussalam,14,2010,2023,14,0,,True
1,baseline_theory_fit,Cambodia,14,2010,2023,14,0,,True
2,baseline_theory_fit,Indonesia,14,2010,2023,14,0,,True
3,baseline_theory_fit,Lao PDR,1,2010,2010,1,0,,True
4,baseline_theory_fit,Malaysia,14,2010,2023,14,0,,True
5,baseline_theory_fit,Philippines,10,2010,2019,10,0,,True
6,baseline_theory_fit,Singapore,11,2010,2020,11,0,,True
7,baseline_theory_fit,Thailand,14,2010,2023,14,0,,True
8,baseline_theory_fit,Timor-Leste,14,2010,2023,14,0,,True
9,baseline_theory_fit,Viet Nam,13,2010,2022,13,0,,True


,variable,country,usable_years,first_usable_year,last_usable_year,window_length_years,internal_missing_years,internal_missing_year_list,is_contiguous_window
0,broad_money_pct_gdp,Brunei Darussalam,14,2010,2023,14,0,,True
1,broad_money_pct_gdp,Cambodia,14,2010,2023,14,0,,True
2,broad_money_pct_gdp,Indonesia,14,2010,2023,14,0,,True
3,broad_money_pct_gdp,Lao PDR,1,2010,2010,1,0,,True
4,broad_money_pct_gdp,Malaysia,14,2010,2023,14,0,,True
5,broad_money_pct_gdp,Myanmar,11,2010,2020,11,0,,True
6,broad_money_pct_gdp,Philippines,13,2010,2022,13,0,,True
7,broad_money_pct_gdp,Singapore,11,2010,2020,11,0,,True
8,broad_money_pct_gdp,Thailand,14,2010,2023,14,0,,True
9,broad_money_pct_gdp,Timor-Leste,14,2010,2023,14,0,,True


,issue,current_status,risk,recommended_response
0,complete_case_unbalanced_panel,complete-case samples differ across model spec...,coefficients may reflect country/year coverage...,"report country-year window audits, keep main m..."
1,short_country_series,flag country-spec samples with fewer than 8 us...,country fixed effects become weakly informed f...,avoid making country-specific claims and test ...
2,control_imputation_scope,only controls are interpolated; FDI and moneta...,key-variable missingness still drives sample loss,make imputation flags visible and avoid imputi...


## 09 Export Processed Panel And Audit Workbook

Write the clean panel and preprocessing artifacts used by downstream notebooks.

In [30]:
clean_panel.to_csv(PROCESSED_DIR / 'clean_panel.csv', index=False)
verification_table.to_csv(OUTPUTS_DIR / 'verification_table.csv', index=False)
transformation_audit.to_csv(OUTPUTS_DIR / 'transformation_audit.csv', index=False)
coverage_by_variable.to_csv(OUTPUTS_DIR / 'coverage_by_variable.csv')
coverage_by_country.to_csv(OUTPUTS_DIR / 'coverage_by_country.csv')
panel_grid.to_csv(OUTPUTS_DIR / 'panel_grid.csv')
review_flags.to_csv(OUTPUTS_DIR / 'review_flags.csv', index=False)
winsorization_thresholds.to_csv(OUTPUTS_DIR / 'winsorization_thresholds.csv', index=False)
spec_sample_summary.to_csv(OUTPUTS_DIR / 'spec_sample_summary.csv', index=False)
spec_sample_country_counts.to_csv(OUTPUTS_DIR / 'spec_sample_country_counts.csv', index=False)
spec_vif_table.to_csv(OUTPUTS_DIR / 'specification_vif.csv', index=False)
spec_vif_summary.to_csv(OUTPUTS_DIR / 'specification_vif_summary.csv', index=False)
base_panel_balance_by_country.to_csv(OUTPUTS_DIR / 'base_panel_balance_by_country.csv', index=False)
spec_panel_balance_summary.to_csv(OUTPUTS_DIR / 'spec_panel_balance_summary.csv', index=False)
spec_panel_balance_by_country.to_csv(OUTPUTS_DIR / 'spec_panel_balance_by_country.csv', index=False)
variable_country_coverage_windows.to_csv(OUTPUTS_DIR / 'variable_country_coverage_windows.csv', index=False)
panel_balance_method_note.to_csv(OUTPUTS_DIR / 'panel_balance_method_note.csv', index=False)
missingness_mechanism_diagnostics.to_csv(OUTPUTS_DIR / 'missingness_mechanism_diagnostics.csv', index=False)
missingness_handling_summary.to_csv(OUTPUTS_DIR / 'missingness_handling_summary.csv', index=False)
control_imputation_log.to_csv(OUTPUTS_DIR / 'control_imputation_log.csv', index=False)

variable_audit = missingness_handling_summary.merge(
    coverage_by_variable.reset_index().rename(columns={'index': 'variable'})[['variable', 'non_missing', 'missing', 'missing_rate']],
    on=['variable', 'missing_rate'],
    how='left',
)
variable_audit.to_csv(OUTPUTS_DIR / 'variable_audit.csv', index=False)

with pd.ExcelWriter(OUTPUTS_DIR / 'preprocessing_outputs.xlsx', engine='openpyxl') as writer:
    verification_table.to_excel(writer, sheet_name='verification', index=False)
    transformation_audit.to_excel(writer, sheet_name='transformations', index=False)
    variable_audit.to_excel(writer, sheet_name='variable_audit', index=False)
    coverage_by_country.to_excel(writer, sheet_name='coverage_by_country')
    coverage_by_variable.to_excel(writer, sheet_name='coverage_by_variable')
    review_flags.to_excel(writer, sheet_name='review_flags', index=False)
    winsorization_thresholds.to_excel(writer, sheet_name='winsor_thresholds', index=False)
    spec_sample_summary.to_excel(writer, sheet_name='spec_sample_summary', index=False)
    spec_sample_country_counts.to_excel(writer, sheet_name='sample_country_counts', index=False)
    spec_vif_summary.to_excel(writer, sheet_name='spec_vif_summary', index=False)
    spec_vif_table.to_excel(writer, sheet_name='spec_vif_detail', index=False)
    spec_panel_balance_summary.to_excel(writer, sheet_name='panel_balance_summary', index=False)
    spec_panel_balance_by_country.to_excel(writer, sheet_name='panel_balance_country', index=False)
    variable_country_coverage_windows.to_excel(writer, sheet_name='variable_country_windows', index=False)
    panel_balance_method_note.to_excel(writer, sheet_name='panel_method_note', index=False)
    missingness_mechanism_diagnostics.to_excel(writer, sheet_name='missingness_mechanism', index=False)
    missingness_handling_summary.to_excel(writer, sheet_name='missingness_handling', index=False)
    control_imputation_log.to_excel(writer, sheet_name='imputation_log', index=False)

print('Saved processed data and preprocessing outputs.')

Saved processed data and preprocessing outputs.
